# Scenario

> ⏱ ~25 min

This capstone returns to the Lab 1 travel concierge. Lab 1 taught the build → evaluate → analyse → iterate loop; Lab 2 adds specialised Cohere capabilities you can plug into that loop. For bridge hooks, see `../lab-1-foundry-maf/bridge/travel-capability-hooks.md`.

A **capstone** is a final practice activity that combines several skills. Here, you will add search, relationship mapping, ranking, and chart explanation ideas to the travel-concierge story.


## 2. What you will do

1. Build an embedding index over `try.travel/corpus/`.
2. Model a tiny business graph over travellers, suppliers, policies, and routes.
3. Rerank 20 hotel candidates using a nuanced preference.
4. Read travel chart data from `try.travel/charts.html` and ask Command A to explain it.
5. Explore per-capability sandbox experiments keyed to each focused notebook.

Each step shows a different way to make the concierge smarter. The next code cells are small on purpose, so you can see the idea before thinking about production scale.

---


## 3. Setup

An **embedding** is a number fingerprint that captures meaning. An embedding index stores those fingerprints so you can search by “trip vibe” instead of exact words. **Rerank** is the concierge’s final judgement pass that sorts an existing shortlist by a detailed preference.

A **graph** maps relationships, such as which traveler uses which route or which supplier fits which policy. The setup cell loads shared helpers and environment values so each capability can use the right deployment.


In [ ]:
from pathlib import Path
import os
import numpy as np
from dotenv import load_dotenv
import cohere
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load the Lab 0 .env whether the notebook is run from the repo root or this folder.
# Locate cohere/ portably (do not hardcode the repo name; learners may fork+rename).
_COHERE_DIR = Path.cwd().resolve()
if _COHERE_DIR.name != 'cohere':
    for _p in (_COHERE_DIR, *_COHERE_DIR.parents):
        if (_p / 'cohere' / 'sample.env').exists():
            _COHERE_DIR = _p / 'cohere'
            break
ENV_PATH = _COHERE_DIR / '.env'
load_dotenv(ENV_PATH, override=True)

# Lab 0 setup-env.sh writes these deployment names to cohere/.env; sample.env documents each key.
PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
AZURE_AI_ENDPOINT = os.environ["AZURE_AI_ENDPOINT"]
EMBED_MODEL = os.environ["EMBED_V4_DEPLOYMENT"]
CHAT_MODEL = os.environ["COMMAND_A_DEPLOYMENT"]
credential = DefaultAzureCredential()

# Cohere's native SDK preserves Embed v4's input_type knob, which the OpenAI-compatible shape does not expose.
cohere_base_url = f"{AZURE_AI_ENDPOINT.rstrip('/')}/providers/cohere"
embed_client = cohere.ClientV2(
    api_key=lambda: credential.get_token("https://cognitiveservices.azure.com/.default").token,
    base_url=cohere_base_url,
)

# Embed v4 uses input_type to tune vectors for stored documents versus user queries.
def embed_texts(texts, input_type="search_document"):
    """Return a NumPy matrix from the Foundry Cohere Embed v4 deployment."""
    response = embed_client.embed(
        model=EMBED_MODEL,
        texts=list(texts),
        input_type=input_type,
        embedding_types=["float"],
    )
    return np.array(response.embeddings.float_, dtype=np.float32)

# Normalize before the dot product so similarity depends on direction, not vector length.
def cosine_matrix(a, b):
    """Cosine similarity for two embedding matrices."""
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a @ b.T

import json
import re

# Project-scoped OpenAI-compatible client matches the Lab 1 chat pattern and shares auth with the agent calls.
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
chat_client = project.get_openai_client()


## 4. Embed — search by trip vibe

Build a simple in-memory index over the travel corpus using Cohere Embed v4. In-memory means the index lives only while the notebook runs, which is enough for learning.

The next code cell turns each travel document into an embedding and compares it with the query embedding. This helps the concierge find trips with similar meaning even when the exact words differ.


In [ ]:
from pathlib import Path

corpus_dir = Path("try.travel/corpus")
paths = sorted(corpus_dir.glob("*.md"))
docs = [path.read_text(encoding="utf-8") for path in paths]

# Store corpus pages as search_document vectors; this is the travel brochure shelf.
index_vectors = embed_texts(docs, input_type="search_document")
query = "trips that feel like a beach getaway"
# User intent is embedded as search_query so Cohere optimizes it for retrieval matching.
query_vector = embed_texts([query], input_type="search_query")
scores = cosine_matrix(query_vector, index_vectors)[0]

# Highest cosine scores are the itinerary pages closest to the requested travel vibe.
for idx in scores.argsort()[::-1][:5]:
    print(f"{scores[idx]:.3f} | {paths[idx].name}")
    print(docs[idx].splitlines()[1][:180], "\n")


## 5. Business graphs — reason over relationships

A graph lets the agent answer with evidence: traveller preference connects to supplier attributes, policy rules, and routes. Evidence matters because travel decisions often depend on more than one fact.

The next code cell builds a tiny graph in plain Python. It shows the shape of the idea without needing a graph database.


In [ ]:
# A tiny graph makes preference matching tangible without requiring a graph database.
travellers = {
    "Asha": {"prefers": ["quiet", "independent hotel"], "destination": "Seattle", "purpose": "conference"},
    "Mateo": {"prefers": ["low carbon", "rail"], "destination": "Vancouver", "purpose": "workshop"},
}
suppliers = {
    "Quiet Harbor Inn": {"city": "Seattle", "type": "independent hotel", "walk_minutes": 5, "noise": "low"},
    "MegaStay Downtown": {"city": "Seattle", "type": "chain hotel", "walk_minutes": 1, "noise": "medium"},
    "Northwest Rail": {"route": "Seattle to Vancouver", "carbon": "low", "refundable": True},
}
policies = [
    {"rule": "manager approval", "when": "hotel rate over 500"},
    {"rule": "prefer rail", "when": "route is under 4 hours and available"},
]

traveller = travellers["Asha"]
matches = []
# Score suppliers by the attributes that matter to the traveller, then sort recommendations.
for name, supplier in suppliers.items():
    if supplier.get("city") == traveller["destination"]:
        score = int(supplier.get("noise") == "low") + int("independent" in supplier.get("type", ""))
        matches.append((score, name, supplier))

for score, name, supplier in sorted(matches, reverse=True):
    print(name, "score", score, supplier)
print("Policy context:", policies)


## 6. Rerank — sharpen a 20-hotel shortlist

Keyword or embedding retrieval can produce a broad candidate list. Rerank sorts it for the traveller’s nuanced preference: quiet, walkable, and not a chain.

The next code cell sends the query and hotel candidates to the rerank deployment. It returns a better order, not new hotel data, so rerank works best after you already have candidates.


In [ ]:
from pathlib import Path
import os
import cohere
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

# Reload .env here so this standalone rerank section works even if earlier cells were skipped.
# Locate cohere/ portably (do not hardcode the repo name; learners may fork+rename).
_COHERE_DIR = Path.cwd().resolve()
if _COHERE_DIR.name != 'cohere':
    for _p in (_COHERE_DIR, *_COHERE_DIR.parents):
        if (_p / 'cohere' / 'sample.env').exists():
            _COHERE_DIR = _p / 'cohere'
            break
ENV_PATH = _COHERE_DIR / '.env'
load_dotenv(ENV_PATH, override=True)

RERANK_MODEL = os.environ["RERANK_DEPLOYMENT"]
# Cohere ClientV2 talks to Foundry's /providers/cohere route while still using Entra auth.
endpoint = os.environ["AZURE_AI_ENDPOINT"].rstrip("/") + "/providers/cohere"
token = DefaultAzureCredential().get_token("https://cognitiveservices.azure.com/.default").token
co = cohere.ClientV2(**{"base_url": endpoint, "api_key": token, "timeout": 300})

# Rerank evaluates query-document relevance directly, ideal after a broad keyword/vector candidate set.
def rerank(query, documents, top_n=3):
    """Call the Foundry-hosted Cohere rerank deployment and return ranked results."""
    return co.rerank(model=RERANK_MODEL, query=query, documents=documents, top_n=top_n)

def show_rerank(query, documents, top_n=3):
    result = rerank(query, documents, top_n=top_n)
    print(f"Query: {query}\n")
    for rank, item in enumerate(result.results, start=1):
        print(f"{rank}. score={item.relevance_score:.3f} | {documents[item.index]}")


# Candidate descriptions intentionally overlap so rerank must balance quiet, walkable, and non-chain signals.
hotel_candidates = [
    "Quiet Harbor Inn — independent boutique, 5-minute walk to convention center, quiet rooms",
    "MegaStay Downtown — chain, attached to convention center, busy lobby",
    "Pioneer House — independent historic hotel, 12-minute walk, quiet courtyard",
    "Airport Value Lodge — chain, airport shuttle, quiet but 45 minutes away",
    "Nightlife Suites — independent, 6-minute walk, loud entertainment district",
    "Maple Conference Hotel — regional chain, 4-minute walk, quiet executive floor",
    "Harbor Hostel Private Rooms — independent, 18-minute walk, budget, shared halls",
    "Convention Grand — luxury chain, 2-minute walk, high-traffic lobby",
    "Cedar House — independent, 9-minute walk, low-noise guarantee",
    "Railway Rooms — independent, 15-minute walk, next to train station noise",
    "Garden Court — boutique independent, 7-minute walk, interior courtyard",
    "Skyline Suites — chain, 10-minute walk, quiet upper floors",
    "Old Port Inn — independent, 20-minute walk, quiet waterfront",
    "Budget Center Stay — chain, 8-minute walk, thin walls",
    "Library Hotel — independent, 11-minute walk, quiet reading lounge",
    "Arena Hotel — chain, 6-minute walk, event-night noise",
    "Elm Street Rooms — independent, 13-minute walk, residential area",
    "Executive Tower — chain, 3-minute walk, quiet concierge floor",
    "Artists Loft Hotel — independent, 14-minute walk, lively lobby",
    "North Pier Lodge — independent, 16-minute walk, quiet but limited breakfast",
]
preference = "quiet, walking distance to the convention center, not a chain"
show_rerank(preference, hotel_candidates, top_n=5)


## 7. Multimodal / chart understanding — explain the travel programme

**Multimodal** means working with more than one kind of input, such as text plus images or chart data. The chart file is standalone HTML with embedded SVG and JSON data.

Here the agent reads the chart data and explains what a travel manager should notice. This is useful when business users need a plain-language summary, not raw numbers.


In [ ]:
# Pull the embedded JSON out of the chart page so the model reasons over data, not pixels.
chart_html = Path("try.travel/charts.html").read_text(encoding="utf-8")
match = re.search(r'<script type="application/json" id="travel-chart-data">(.*?)</script>', chart_html, re.S)
chart_data = json.loads(match.group(1))

question = "Which department peaks in Q3, and which route has the highest booking volume? Give one travel-programme action."
# Send compact chart data plus a specific question to get an analyst-style travel-program action.
response = chat_client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise corporate travel analyst."},
        {"role": "user", "content": f"Chart data: {json.dumps(chart_data)}\n\nQuestion: {question}"},
    ],
    max_tokens=250,
)
print(response.choices[0].message.content)


## 8. Sandbox: per-capability travel experiments

Use this section as a scratchpad that connects each focused Lab 2 notebook back to the Lab 1 travel-concierge scenario. Run the setup cells above first, then try the experiments below in order or jump to the capability you want to explore.

A sandbox is a safe place to change inputs and observe results. Do not worry about finding the perfect answer; focus on how each capability changes the concierge behavior.


### From lab-2a-embed — Embed v4 semantic travel search

Use the same Embed v4 helper from the focused notebook to search the travel corpus by trip intent rather than exact keywords.

**Semantic** means “based on meaning.” The next exercise asks the concierge to match the idea of a trip, not just the exact words in the files.


Reframe embeddings for the Lab 1 concierge: instead of matching exact words like “beach,” retrieve trips that *feel* like a beach getaway, even if the document says “waterfront,” “sunset,” or “resort.”

Exercises:
1. Change `query` to describe your ideal trip.
2. Increase `top_k` and inspect whether results become less focused.
3. Add a new file to `try.travel/corpus/` and rerun the cell.


In [ ]:
from pathlib import Path

corpus_dir = Path("try.travel/corpus")
travel_docs = sorted(corpus_dir.glob("*.md"))
texts = [path.read_text(encoding="utf-8") for path in travel_docs]

query = "trips that feel like a quiet beach getaway with no red-eye flights"  # TODO: try your own travel vibe
top_k = 5

# Recompute vectors after learners edit the query/corpus so the retrieval demo stays self-contained.
doc_vectors = embed_texts(texts, input_type="search_document")
# search_query tells Embed v4 this text is the traveller's request, not a document to index.
query_vector = embed_texts([query], input_type="search_query")
similarities = cosine_matrix(query_vector, doc_vectors)[0]

for idx in similarities.argsort()[::-1][:top_k]:
    print(f"{similarities[idx]:.3f} | {travel_docs[idx].name}")
    print(texts[idx].splitlines()[1][:160], "\n")


### From lab-2b-embed — Business graph reasoning

Apply the business-graph mindset to traveller, supplier, policy, and route relationships from the concierge scenario.

The graph helps explain why an answer is safe. For example, a traveler can connect to a route, the route can connect to a supplier, and the supplier can connect to a policy rule.


In a travel concierge, the same graph mindset can connect travellers, suppliers, policies, and routes. You do not need images for the exercise; model the relationships directly and ask what an agent could infer.

Exercises:
1. Add one new route or supplier relationship.
2. Change the traveller profile and rerun the policy check.
3. Explain which extra edge would make the answer more confident.


In [ ]:
# This hand-built graph shows how explicit relationships can explain a recommendation.
travel_graph = {
    "Asha": {"travels_to": ["Seattle"], "prefers": ["quiet hotel", "independent property"]},
    "Seattle": {"served_by": ["Northwest Rail", "Contoso Air"], "near": ["Convention Center"]},
    "Quiet Harbor Inn": {"type": "independent hotel", "near": ["Convention Center"], "noise": "low"},
    "Policy": {"requires": ["refundable fare for legal meetings", "manager approval over $500/night"]},
}

# TODO: add a relationship, for example another hotel or route.
traveller = "Asha"
need = "quiet hotel near the Convention Center that is not a chain"

candidate = "Quiet Harbor Inn"
# Collect human-readable reasons so the recommendation is auditable, not just a score.
reasons = []
if "quiet hotel" in travel_graph[traveller]["prefers"] and travel_graph[candidate]["noise"] == "low":
    reasons.append("matches quiet preference")
if "Convention Center" in travel_graph[candidate]["near"]:
    reasons.append("walkable to the convention center")
if travel_graph[candidate]["type"] == "independent hotel":
    reasons.append("not a chain")

print(f"Need: {need}")
print(f"Recommendation: {candidate}")
print("Reasons:", "; ".join(reasons))


### From lab-2c-rerank-getting-started — Rerank a hotel shortlist

Start with a rough hotel candidate list, then use rerank to sort by the traveller's nuanced preference.

Nuanced means detailed and specific. “Good hotel” is vague. “Quiet, walkable, and not a chain” gives rerank clearer signals.


Use rerank after an initial retrieval step. Imagine a keyword search returned these hotels; now rerank based on what the traveller really cares about.

Exercises:
1. Change `traveller_preference` to emphasize budget, brand, or quiet.
2. Add a hotel with conflicting evidence.
3. Compare the top result before and after rerank.


In [ ]:
# Edit the preference string to watch rerank reorder the same hotel shortlist.
traveller_preference = "quiet, walking distance to the convention center, not a chain"  # TODO: edit this
hotel_candidates = [
    "Quiet Harbor Inn: independent boutique hotel, 5-minute walk to convention center, low-noise rooms",
    "MegaStay Downtown: large chain hotel, attached to convention center, busy lobby",
    "Airport Value Lodge: budget chain near airport, shuttle required, quiet rooms",
    "Pioneer House: historic independent hotel, 12-minute walk, limited breakfast",
    "Nightlife Suites: independent hotel near music venues, 6-minute walk, loud on weekends",
]
# top_n limits the display to the few options a traveller can reasonably compare.
show_rerank(traveller_preference, hotel_candidates, top_n=3)


### From optional-lab-rerank_structured_data — Rerank structured travel records

Turn structured travel records into rankable documents so a plain-language policy or preference can sort them.

Structured records are data with fields, like hotel name, price, distance, and refund rules. The next exercise shows how to serialize those fields so rerank can compare them.


Travel systems are full of structured records: bookings, policies, suppliers, and preferences. Rerank can sort those records by a plain-language need.

Exercises:
1. Add a candidate hotel or flight row.
2. Change the query to require accessibility, budget, or sustainability.
3. Inspect which fields caused the top result to win.


In [ ]:
import yaml

# YAML records preserve structured fields while giving rerank natural-language evidence.
travel_records = [
    {"hotel": "Quiet Harbor Inn", "brand": "independent", "walk_minutes": 5, "noise": "low", "rate": 420},
    {"hotel": "MegaStay Downtown", "brand": "chain", "walk_minutes": 1, "noise": "medium", "rate": 390},
    {"hotel": "Pioneer House", "brand": "independent", "walk_minutes": 12, "noise": "low", "rate": 310},
    {"hotel": "Airport Value Lodge", "brand": "chain", "walk_minutes": 45, "noise": "low", "rate": 180},
]
# Convert each supplier row into one document string before reranking.
travel_docs = [yaml.safe_dump(record, sort_keys=False) for record in travel_records]
query = "quiet independent hotel within walking distance of the convention center under 450 dollars"  # TODO: edit
show_rerank(query, travel_docs, top_n=3)


### From optional-lab-rerank_wikipedia_search — Rerank a local travel corpus

Use the local travel corpus as a first-pass lexical search source, then rerank the candidates for concierge intent.

Lexical search matches words directly. It is fast and simple, but it can miss meaning. Rerank helps after that first pass by sorting the found items against the user’s real intent.


Use the local travel corpus as the lexical candidate source. In a real app, the first pass could be a database search, BM25, or embedding retrieval; rerank then sharpens the order.

Exercises:
1. Replace the keyword filter with your own quick search.
2. Change the rerank query to include a negation.
3. Compare raw candidate order to reranked order.


In [ ]:
from pathlib import Path

query = "quiet independent hotel near a convention center, not airport"  # TODO: edit
keyword = "hotel"  # TODO: try "beach", "family", or "policy"
# Keyword filtering creates a cheap candidate pool; rerank then handles nuanced preference ordering.
travel_docs = []
for path in sorted(Path("try.travel/corpus").glob("*.md")):
    text = path.read_text(encoding="utf-8")
    if keyword.lower() in text.lower():
        travel_docs.append(f"{path.name}: {text}")

print("Raw keyword candidates:")
for doc in travel_docs[:5]:
    print("-", doc.splitlines()[0])

# Use min(...) so the demo still works when the keyword returns fewer than three candidates.
show_rerank(query, travel_docs, top_n=min(3, len(travel_docs)))


## 9. What you learned

1. Embed, graph reasoning, rerank, structured-data ranking, corpus search, and chart analysis each address a different travel-concierge capability.
2. The focused Lab 2 notebooks can stay vendor-faithful while this sandbox adapts each capability to the Lab 1 travel scenario.
3. A complete travel concierge combines retrieval breadth, relationship evidence, nuanced ranking, and business-signal analysis.
